# LeetCode 639 — Decode Ways II bằng Python

- Link: https://leetcode.com/problems/decode-ways-ii/
- Độ khó: **Hard**
- Chủ đề: Quy hoạch động (DP), Đếm tổ hợp, Phép chia lấy dư (modulo), Máy trạng thái

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def numDecodings(self, s: str) -> int:

```

## 1. Giới thiệu & mục tiêu học

Bài này là bản "nâng cấp" của **Decode Ways (LC 91)**: chuỗi giờ có thêm ký tự đại diện `*`. Ý tưởng DP vẫn y hệt — "chữ cái cuối dùng **1** chữ số hay **2** chữ số?" — nhưng mỗi bước phải **đếm số khả năng** thay vì chỉ kiểm tra hợp lệ. Cái khó nằm ở việc liệt kê đủ và đúng các trường hợp với `*`.

Sau notebook này bạn sẽ:

1. Nắm lại DP của Decode Ways I: `dp[i] = dp[i-1]·(1 chữ số hợp lệ) + dp[i-2]·(2 chữ số hợp lệ)`.
2. Biết tách bài toán thành hai hàm đếm nhỏ: `one(c)` và `two(a, b)` — số cách một/hai ký tự tạo thành một chữ cái.
3. Viết lần lượt: brute force thay `*`, đệ quy có nhớ, DP mảng, DP hai biến `O(1)` bộ nhớ, và DP máy trạng thái.
4. Hiểu vì sao phải lấy modulo `10^9 + 7` **ở mỗi bước** và làm thế nào để không quên.
5. Kiểm chứng mọi cách bằng brute force trên input ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Một thông điệp gồm chữ cái `A`–`Z` được mã hóa: `A → "1"`, `B → "2"`, ..., `Z → "26"`. Để giải mã, ta chia chuỗi chữ số thành các nhóm, mỗi nhóm là một số từ `1` tới `26` **không có số 0 đứng đầu** (`"06"` không hợp lệ, `"6"` thì hợp lệ).

Ngoài chữ số, chuỗi `s` có thể chứa `*` — đại diện cho **một chữ số bất kỳ từ `1` đến `9`** (không bao gồm `0`). Đếm **tổng số cách giải mã** (tính trên mọi cách thay `*`), trả về kết quả **modulo `10^9 + 7`**.

**Ví dụ 1**

```text
Input:  s = "*"
Output: 9
Giải thích: "*" là "1".."9", mỗi cái giải mã đúng 1 cách (A..I).
```

**Ví dụ 2**

```text
Input:  s = "1*"
Output: 18
Giải thích: "1*" là "11".."19". Mỗi chuỗi có 2 cách ("11" = "AA" hoặc "K"), 9 · 2 = 18.
```

**Ví dụ 3**

```text
Input:  s = "2*"
Output: 15
Giải thích: "21".."26" mỗi chuỗi 2 cách (6 · 2 = 12); "27".."29" mỗi chuỗi 1 cách (3). Tổng 15.
```

**Ràng buộc**

- `1 <= s.length <= 10^5`
- `s[i]` là chữ số `'0'`–`'9'` hoặc `'*'`.

In [1]:
def decode_ways_I(digits):
    """Decode Ways I (không có *): đếm số cách giải mã chuỗi chữ số."""
    n = len(digits)
    dp = [0] * (n + 1)
    dp[0] = 1
    for i in range(1, n + 1):
        if digits[i - 1] != '0':
            dp[i] += dp[i - 1]
        if i >= 2 and digits[i - 2] != '0' and int(digits[i - 2:i]) <= 26:
            dp[i] += dp[i - 2]
    return dp[n]

print("11 ->", decode_ways_I("11"), "| 27 ->", decode_ways_I("27"), "| 226 ->", decode_ways_I("226"), "| 06 ->", decode_ways_I("06"))
print("Ví dụ 2 bằng tay:", sum(decode_ways_I("1" + d) for d in "123456789"))
print("Ví dụ 3 bằng tay:", sum(decode_ways_I("2" + d) for d in "123456789"))

11 -> 2 | 27 -> 1 | 226 -> 3 | 06 -> 0
Ví dụ 2 bằng tay: 18
Ví dụ 3 bằng tay: 15


Cell trên kiểm chứng ví dụ 2 và 3 bằng cách thay `*` thủ công rồi cộng lại — đây cũng chính là ý tưởng brute force ở mục 5.

## 3. Kiến thức nền

### 3.1. Phép chia lấy dư và vì sao lấy mod ở mỗi bước

Kết quả có thể cực lớn (cỡ `9^100000`). Python có số nguyên vô hạn nên **không tràn số**, nhưng số càng lớn thì phép cộng/nhân càng chậm. Ta lấy `% MOD` ngay sau mỗi phép tính để số luôn nhỏ hơn `10^9 + 7`. Tính chất: `(a + b) % m = ((a % m) + (b % m)) % m`, `(a · b) % m = ((a % m) · (b % m)) % m`.

In [2]:
MOD = 10**9 + 7
a, b = 987654321987, 123456789123
print((a + b) % MOD, ((a % MOD) + (b % MOD)) % MOD)
print((a * b) % MOD, ((a % MOD) * (b % MOD)) % MOD)
print("9^50 có", len(str(9**50)), "chữ số; mod ->", pow(9, 50, MOD))

111103333 111103333
187830321 187830321
9^50 có 48 chữ số; mod -> 886041711


### 3.2. So sánh ký tự chữ số

Các ký tự `'0'`..`'9'` có mã liên tiếp, nên so sánh chuỗi một ký tự hoạt động như so sánh số: `'3' <= '6'` là `True`. Lưu ý `'*'` có mã nhỏ hơn `'0'`, nên **luôn xử lý `'*'` riêng trước** khi so sánh.

In [3]:
print('3' <= '6', '7' <= '6', '0' < '1')
print("mã ký tự:", ord('*'), ord('0'), ord('9'))
print("'*' <= '6' ?", '*' <= '6', "<- nguy hiểm nếu quên xử lý * riêng!")

True False True
mã ký tự: 42 48 57
'*' <= '6' ? True <- nguy hiểm nếu quên xử lý * riêng!


### 3.3. `itertools.product` — liệt kê mọi cách thay `*`

`product("123456789", repeat=m)` sinh mọi bộ `m` chữ số — dùng cho brute force.

In [4]:
from itertools import product
subs = list(product("123", repeat=2))
print(len(subs), subs[:5])

def expand(s):
    """Sinh mọi chuỗi chữ số thu được khi thay mỗi * bằng 1..9."""
    stars = [i for i, c in enumerate(s) if c == '*']
    for choice in product("123456789", repeat=len(stars)):
        t = list(s)
        for pos, d in zip(stars, choice):
            t[pos] = d
        yield ''.join(t)

print(list(expand("1*"))[:4], "...", len(list(expand("**"))))

9 [('1', '1'), ('1', '2'), ('1', '3'), ('2', '1'), ('2', '2')]
['11', '12', '13', '14'] ... 81


### 3.4. Đệ quy có nhớ với `functools.lru_cache`

`@lru_cache(maxsize=None)` lưu kết quả hàm theo tham số, gọi lại thì trả ngay. Với đệ quy sâu (n = 10^5) cần cẩn thận giới hạn đệ quy của Python (mặc định ~1000) — vì vậy bản nộp sẽ dùng vòng lặp.

In [5]:
from functools import lru_cache
import sys

@lru_cache(maxsize=None)
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

print(fib(80), "| recursion limit:", sys.getrecursionlimit())

23416728348467685 | recursion limit: 1000


### 3.5. DP "cuộn" bằng hai biến

Khi `dp[i]` chỉ phụ thuộc `dp[i-1]` và `dp[i-2]`, giữ hai biến `prev2, prev1` và cập nhật bằng **gán đồng thời** `prev2, prev1 = prev1, new`.

In [6]:
prev2, prev1 = 0, 1            # fib(0), fib(1)
for _ in range(9):
    prev2, prev1 = prev1, prev1 + prev2
print("fib(10) =", prev1)

fib(10) = 55


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Nhìn vào chữ cái cuối cùng

Mỗi cách giải mã `s[0..i-1]` kết thúc bằng một chữ cái dùng **1 ký tự cuối** hoặc **2 ký tự cuối**. Nếu gọi `dp[i]` = số cách giải mã `i` ký tự đầu (tính cả mọi cách thay `*`), thì:

```text
dp[i] = one(s[i-1]) · dp[i-1]  +  two(s[i-2], s[i-1]) · dp[i-2]
```

- `one(c)` = số chữ số `c` có thể là mà tạo thành chữ cái (1..9).
- `two(a, b)` = số cặp `(a, b)` có thể là mà tạo thành số 10..26.

Vì sao **nhân**? Phần đầu có `dp[i-1]` cách, chữ cái cuối có `one(...)` lựa chọn, hai phần độc lập → quy tắc nhân.

### 4.2. Bảng `one(c)`

| c | các giá trị | one(c) |
|---|---|---|
| `'0'` | 0 (không hợp lệ đứng một mình) | 0 |
| `'1'`..`'9'` | chính nó | 1 |
| `'*'` | 1..9 | 9 |

### 4.3. Bảng `two(a, b)` — phần dễ sai nhất

| a | b | các số hợp lệ | two |
|---|---|---|---|
| `'1'` | chữ số | 10..19 | 1 |
| `'1'` | `'*'` | 11..19 | **9** |
| `'2'` | `'0'`..`'6'` | 20..26 | 1 |
| `'2'` | `'7'`..`'9'` | không | 0 |
| `'2'` | `'*'` | 21..26 | **6** |
| `'*'` | `'0'`..`'6'` | 1b và 2b | **2** |
| `'*'` | `'7'`..`'9'` | chỉ 1b | **1** |
| `'*'` | `'*'` | 11..19, 21..26 | **15** |
| khác (`'0'`, `'3'`..`'9'`) | bất kỳ | không | 0 |

Chú ý: `*` **không** là 0, nên `'1*'` chỉ có 11..19 (9 số, không có 10), `'**'` không có 10 và 20.

### 4.4. Tính tay `s = "1*"`

- `dp[0] = 1` (chuỗi rỗng có 1 cách: không làm gì).
- `i = 1`, ký tự `'1'`: `dp[1] = one('1') · dp[0] = 1`.
- `i = 2`, ký tự `'*'`: `dp[2] = one('*') · dp[1] + two('1','*') · dp[0] = 9·1 + 9·1 = 18`. ✓

### 4.5. Tính tay `s = "**"`

- `dp[1] = 9`.
- `dp[2] = one('*')·dp[1] + two('*','*')·dp[0] = 9·9 + 15·1 = 96`.

Kiểm chứng: 81 chuỗi "xy" (x, y ∈ 1..9), mỗi chuỗi có 1 cách tách rời, cộng thêm 15 chuỗi mà "xy" ≤ 26 cho cách thứ hai → 81 + 15 = 96.

In [7]:
def one(c):
    if c == '*':
        return 9
    return 0 if c == '0' else 1

def two(a, b):
    if a == '*':
        if b == '*':
            return 15
        return 2 if b <= '6' else 1
    if a == '1':
        return 9 if b == '*' else 1
    if a == '2':
        if b == '*':
            return 6
        return 1 if b <= '6' else 0
    return 0

# Kiểm tra bảng two bằng cách liệt kê thật
def two_brute(a, b):
    A = "123456789" if a == '*' else a
    B = "123456789" if b == '*' else b
    return sum(1 for x in A for y in B if 10 <= int(x + y) <= 26)

chars = "0123456789*"
for a in chars:
    for b in chars:
        assert two(a, b) == two_brute(a, b), (a, b)
    assert one(a) == sum(1 for x in ("123456789" if a == '*' else a) if x != '0')
print("Bảng one/two khớp liệt kê thật cho cả", len(chars) ** 2, "cặp")
print("two('*','*') =", two('*', '*'), "| two('1','*') =", two('1', '*'), "| two('2','*') =", two('2', '*'), "| two('*','7') =", two('*', '7'))

Bảng one/two khớp liệt kê thật cho cả 121 cặp
two('*','*') = 15 | two('1','*') = 9 | two('2','*') = 6 | two('*','7') = 1


Kỹ thuật quan trọng ở cell trên: **viết bảng bằng tay rồi kiểm tra bằng liệt kê máy**. Với bài nhiều trường hợp như thế này, đây là cách chắc chắn nhất để không sai một ô nào.

## 5. Cách 1 — Brute Force: thay mọi `*` rồi dùng Decode Ways I

**Ý tưởng.** Sinh mọi chuỗi chữ số từ `s` (mỗi `*` thử 9 giá trị), với mỗi chuỗi đếm số cách giải mã bằng DP của LC 91, cộng tất cả lại (mod).

In [8]:
def num_decodings_brute(s):
    total = 0
    for t in expand(s):
        total += decode_ways_I(t)
    return total % MOD

print(num_decodings_brute("*"), num_decodings_brute("1*"), num_decodings_brute("2*"), num_decodings_brute("**"))
assert num_decodings_brute("*") == 9
assert num_decodings_brute("1*") == 18
assert num_decodings_brute("2*") == 15

9 18 15 96


**Độ phức tạp.** Với `m` dấu `*`, có `9^m` chuỗi, mỗi chuỗi `O(n)` → **`O(9^m · n)`**. Chỉ 10 dấu `*` đã là 3,5 tỉ chuỗi. Space `O(n)`.

**Nút thắt.** Các chuỗi khác nhau chia sẻ **tiền tố** giống nhau, nhưng ta tính lại từ đầu. Thay vì cố định `*` trước, hãy **đếm số lựa chọn ngay tại chỗ** khi DP đi qua `*` — đó là `one` và `two`.

In [9]:
import time
for m in (3, 4, 5):
    s = "*1" * m
    t = time.perf_counter()
    r = num_decodings_brute(s)
    print(f"{m} dấu * ({9**m} chuỗi): {r}  {time.perf_counter() - t:.3f}s")

3 dấu * (729 chuỗi): 3842  0.001s
4 dấu * (6561 chuỗi): 73204  0.011s
5 dấu * (59049 chuỗi): 1394924  0.131s


## 6. Bước đột phá

**Insight.** Chữ cái cuối của một cách giải mã chỉ phụ thuộc **1 hoặc 2 ký tự cuối**. Phần còn lại là một bài toán con độc lập (giải mã tiền tố ngắn hơn). Vì vậy:

```text
dp[i] = one(s[i-1]) · dp[i-1] + two(s[i-2], s[i-1]) · dp[i-2]      (mod 1e9+7)
```

`one` và `two` "nén" toàn bộ 9 hay 15 lựa chọn của `*` thành **một con số** — không cần liệt kê.

**Invariant:** sau khi xử lý `i` ký tự, `dp[i]` = tổng số cặp (cách thay `*` trong `s[0..i-1]`, cách tách nhóm hợp lệ), modulo `10^9+7`.

**Quy tắc nếu…thì…:**
- Nếu ký tự cuối là `'0'` thì không thể đứng một mình (`one = 0`); nó chỉ sống sót khi ghép với ký tự trước là `'1'`, `'2'` hoặc `'*'`. Riêng `'*0'` có **2** cách (`10`, `20`) dù `*` không bao giờ là 0 — vì `*` ở đây là chữ số **đầu** của cặp.
- Nếu hai ký tự cuối bắt đầu bằng `'0'` hoặc `'3'`..`'9'` thì không ghép được (`two = 0`).
- Nếu `dp[i-1]` và `dp[i-2]` đều 0 (ví dụ có `"00"`) thì mọi `dp` sau đều 0.

## 7. Các cách tiếp theo

Notebook trình bày **5 cách**:

| # | Cách | Time | Space |
|---|---|---|---|
| 1 | Brute force thay `*` (mục 5) | `O(9^m · n)` | `O(n)` |
| 2 | Đệ quy có nhớ (top-down) | `O(n)` | `O(n)` |
| 3 | DP mảng (bottom-up) | `O(n)` | `O(n)` |
| 4 | DP hai biến | `O(n)` | `O(1)` |
| 5 | DP máy trạng thái `e0, e1, e2` | `O(n)` | `O(1)` |

### 7.1. Cách 2 — Đệ quy có nhớ

**Giải thích.** `f(i)` = số cách giải mã **hậu tố** `s[i:]`. Ở vị trí `i`, chữ cái đầu tiên dùng 1 ký tự (`one(s[i]) · f(i+1)`) hoặc 2 ký tự (`two(s[i], s[i+1]) · f(i+2)`). `f(n) = 1`.

In [10]:
def num_decodings_memo(s):
    n = len(s)

    @lru_cache(maxsize=None)
    def f(i):
        if i == n:
            return 1
        res = one(s[i]) * f(i + 1)
        if i + 1 < n:
            res += two(s[i], s[i + 1]) * f(i + 2)
        return res % MOD

    return f(0)

print(num_decodings_memo("*"), num_decodings_memo("1*"), num_decodings_memo("2*"), num_decodings_memo("**"))
assert num_decodings_memo("1*") == 18 and num_decodings_memo("2*") == 15

9 18 15 96


**Đọc code:**

- `f(i)` nhìn **về phía trước** (hậu tố), ngược với `dp[i]` nhìn về phía sau (tiền tố). Cả hai đều đúng.
- `if i + 1 < n`: chỉ ghép 2 ký tự khi còn đủ 2 ký tự.
- Một tối ưu nhỏ: nếu `one(s[i]) == 0` thì khỏi gọi `f(i+1)` — không bắt buộc.

**Code nộp LeetCode (Cách 2):** với `n = 10^5`, độ sâu đệ quy tới `10^5` → **RecursionError** trừ khi tăng giới hạn (và vẫn có thể tràn stack C). Bản dưới tăng giới hạn chỉ để minh họa; khuyên dùng Cách 3/4.

In [11]:
class SolutionMemo:
    def numDecodings(self, s: str) -> int:
        import sys
        sys.setrecursionlimit(max(10000, 3 * len(s) + 100))
        n = len(s)
        memo = [-1] * (n + 1)

        def f(i):
            if i == n:
                return 1
            if memo[i] >= 0:
                return memo[i]
            res = one(s[i]) * f(i + 1)
            if i + 1 < n:
                res += two(s[i], s[i + 1]) * f(i + 2)
            memo[i] = res % MOD
            return memo[i]

        return f(0)

assert SolutionMemo().numDecodings("1*") == 18
assert SolutionMemo().numDecodings("*" * 500) == num_decodings_memo("*" * 500)
print("SolutionMemo OK (chuỗi ngắn)")

SolutionMemo OK (chuỗi ngắn)


**Độ phức tạp Cách 2:** Time `O(n)` (mỗi `f(i)` tính một lần), Space `O(n)` cho memo + stack.

### 7.2. Cách 3 — DP mảng (bottom-up)

**Giải thích.** Chuyển đệ quy thành vòng lặp trên tiền tố: `dp[0] = 1`, `dp[1] = one(s[0])`, rồi công thức mục 6 cho `i = 2..n`.

In [12]:
def num_decodings_dp(s):
    n = len(s)
    dp = [0] * (n + 1)
    dp[0] = 1
    dp[1] = one(s[0])
    for i in range(2, n + 1):
        dp[i] = (one(s[i - 1]) * dp[i - 1] + two(s[i - 2], s[i - 1]) * dp[i - 2]) % MOD
    return dp[n]

print(num_decodings_dp("*"), num_decodings_dp("1*"), num_decodings_dp("2*"), num_decodings_dp("**"), num_decodings_dp("*0"))
assert num_decodings_dp("**") == 96

9 18 15 96 2


**Đọc code:**

- `dp` có `n + 1` phần tử: `dp[i]` ứng với tiền tố độ dài `i`.
- `dp[1] = one(s[0])`: ký tự đầu tiên chỉ có thể đứng một mình.
- Chỉ số lệch 1: ký tự thứ `i` (đếm từ 1) là `s[i - 1]`. Đây là chỗ hay nhầm nhất.
- `% MOD` ngay trong vòng lặp.

**Code nộp LeetCode (Cách 3):**

In [13]:
class SolutionDP:
    def numDecodings(self, s: str) -> int:
        M = 10**9 + 7

        def one(c):
            return 9 if c == '*' else (0 if c == '0' else 1)

        def two(a, b):
            if a == '*':
                return 15 if b == '*' else (2 if b <= '6' else 1)
            if a == '1':
                return 9 if b == '*' else 1
            if a == '2':
                return 6 if b == '*' else (1 if b <= '6' else 0)
            return 0

        n = len(s)
        dp = [0] * (n + 1)
        dp[0], dp[1] = 1, one(s[0])
        for i in range(2, n + 1):
            dp[i] = (one(s[i - 1]) * dp[i - 1] + two(s[i - 2], s[i - 1]) * dp[i - 2]) % M
        return dp[n]

assert SolutionDP().numDecodings("1*") == 18
print("SolutionDP OK")

SolutionDP OK


**Độ phức tạp Cách 3:** Time `O(n)`, Space `O(n)`.

### 7.3. Cách 4 — DP hai biến (`O(1)` bộ nhớ)

**Giải thích.** `dp[i]` chỉ cần `dp[i-1]` và `dp[i-2]`. Giữ `prev2 = dp[i-2]`, `prev1 = dp[i-1]`.

In [14]:
def num_decodings_rolling(s):
    prev2, prev1 = 1, one(s[0])          # dp[0], dp[1]
    for i in range(1, len(s)):
        cur = (one(s[i]) * prev1 + two(s[i - 1], s[i]) * prev2) % MOD
        prev2, prev1 = prev1, cur
    return prev1

print(num_decodings_rolling("*"), num_decodings_rolling("1*"), num_decodings_rolling("2*"), num_decodings_rolling("**"))
assert num_decodings_rolling("**") == 96

9 18 15 96


**Đọc code:**

- Vòng lặp chạy `i` từ 1 (chỉ số 0-based của ký tự đang xét). `s[i - 1], s[i]` là hai ký tự cuối.
- `prev2, prev1 = prev1, cur`: gán đồng thời — vế phải tính xong mới gán, nên không mất giá trị cũ.
- Trường hợp `len(s) == 1`: vòng lặp không chạy, trả `one(s[0])`.

**Code nộp LeetCode (Cách 4):**

In [15]:
class SolutionRolling:
    def numDecodings(self, s: str) -> int:
        M = 10**9 + 7
        ONE = {'*': 9, '0': 0}
        prev2, prev1 = 1, ONE.get(s[0], 1)
        for i in range(1, len(s)):
            a, b = s[i - 1], s[i]
            o = ONE.get(b, 1)
            if a == '*':
                t = 15 if b == '*' else (2 if b <= '6' else 1)
            elif a == '1':
                t = 9 if b == '*' else 1
            elif a == '2':
                t = 6 if b == '*' else (1 if b <= '6' else 0)
            else:
                t = 0
            prev2, prev1 = prev1, (o * prev1 + t * prev2) % M
        return prev1

assert SolutionRolling().numDecodings("2*") == 15
print("SolutionRolling OK")

SolutionRolling OK


**Độ phức tạp Cách 4:** Time `O(n)`, Space `O(1)`.

### 7.4. Cách 5 — DP máy trạng thái `e0, e1, e2`

**Giải thích.** Một góc nhìn khác (lời giải chính thức của LeetCode): đọc chuỗi từ trái sang phải, sau mỗi ký tự ta ở một trong ba trạng thái:

- `e0`: số cách mà mọi ký tự đã đọc **đã được ghép xong** thành chữ cái.
- `e1`: số cách mà ký tự cuối là **`1` đang chờ** ghép với ký tự kế tiếp (thành `1x`).
- `e2`: số cách mà ký tự cuối là **`2` đang chờ** (thành `2x`, `x <= 6`).

Đọc ký tự `c`:

- Nếu `c == '*'`:
  - `f0 = 9·e0` (đứng một mình) `+ 9·e1` (`11..19`) `+ 6·e2` (`21..26`)
  - `f1 = e0` (chọn `*` = 1 và để chờ), `f2 = e0` (chọn `*` = 2 và để chờ)
- Nếu `c` là chữ số:
  - `f0 = (c != '0')·e0 + e1 + (c <= '6')·e2`
  - `f1 = (c == '1')·e0`, `f2 = (c == '2')·e0`

Khởi đầu `e0 = 1, e1 = e2 = 0`; đáp án `e0` cuối cùng (không được còn ký tự chờ).

In [16]:
def num_decodings_states(s):
    e0, e1, e2 = 1, 0, 0
    for c in s:
        if c == '*':
            f0 = 9 * e0 + 9 * e1 + 6 * e2
            f1 = e0
            f2 = e0
        else:
            f0 = (c > '0') * e0 + e1 + (c <= '6') * e2
            f1 = (c == '1') * e0
            f2 = (c == '2') * e0
        e0, e1, e2 = f0 % MOD, f1, f2
    return e0

print(num_decodings_states("*"), num_decodings_states("1*"), num_decodings_states("2*"), num_decodings_states("**"))
assert num_decodings_states("**") == 96

9 18 15 96


**Đọc code:**

- `(c > '0') * e0`: `True * x = x`, `False * x = 0` — cách viết gọn của "nếu … thì cộng".
- Trạng thái "chờ" cho phép một ký tự **vừa đứng một mình vừa chờ ghép** — hai nhánh được đếm riêng ở `f0` và `f1/f2`.
- Vì sao tương đương Cách 4? `e1 + e2` ở bước sau chính là phần `dp[i-2]` đã "nhân" với lựa chọn chữ số đầu của cặp; ví dụ `two('*', b)` được tách thành "chọn 1 rồi chờ" và "chọn 2 rồi chờ".

**Code nộp LeetCode (Cách 5):**

In [17]:
class SolutionStates:
    def numDecodings(self, s: str) -> int:
        M = 10**9 + 7
        e0, e1, e2 = 1, 0, 0
        for c in s:
            if c == '*':
                e0, e1, e2 = (9 * e0 + 9 * e1 + 6 * e2) % M, e0, e0
            else:
                e0, e1, e2 = ((c > '0') * e0 + e1 + (c <= '6') * e2) % M, (c == '1') * e0, (c == '2') * e0
        return e0

assert SolutionStates().numDecodings("1*") == 18
print("SolutionStates OK")

SolutionStates OK


**Độ phức tạp Cách 5:** Time `O(n)`, Space `O(1)`.

## 8. TRACE từng bước (Cách 3/4)

Trace `s = "1*2*"`:

| i | ký tự | one | cặp | two | dp[i] = one·dp[i-1] + two·dp[i-2] |
|---|---|---|---|---|---|
| 0 | — | — | — | — | 1 |
| 1 | `1` | 1 | — | — | 1·1 = 1 |
| 2 | `*` | 9 | `1*` | 9 | 9·1 + 9·1 = 18 |
| 3 | `2` | 1 | `*2` | 2 | 1·18 + 2·1 = 20 |
| 4 | `*` | 9 | `2*` | 6 | 9·20 + 6·18 = 288 |

In [18]:
def trace_dp(s):
    n = len(s)
    dp = [0] * (n + 1)
    dp[0] = 1
    dp[1] = one(s[0])
    print(f"{'i':>2} | {'kt':>2} | {'one':>3} | {'cặp':>4} | {'two':>3} | dp[i]")
    print(f"{0:>2} | {'-':>2} | {'-':>3} | {'-':>4} | {'-':>3} | 1")
    print(f"{1:>2} | {s[0]:>2} | {one(s[0]):>3} | {'-':>4} | {'-':>3} | {dp[1]}")
    for i in range(2, n + 1):
        o, t = one(s[i - 1]), two(s[i - 2], s[i - 1])
        dp[i] = (o * dp[i - 1] + t * dp[i - 2]) % MOD
        print(f"{i:>2} | {s[i - 1]:>2} | {o:>3} | {s[i - 2:i]:>4} | {t:>3} | {o}·{dp[i - 1]} + {t}·{dp[i - 2]} = {dp[i]}")
    return dp[n]

r = trace_dp("1*2*")
assert r == 288 == num_decodings_brute("1*2*")

 i | kt | one |  cặp | two | dp[i]
 0 |  - |   - |    - |   - | 1
 1 |  1 |   1 |    - |   - | 1
 2 |  * |   9 |   1* |   9 | 9·1 + 9·1 = 18
 3 |  2 |   1 |   *2 |   2 | 1·18 + 2·1 = 20
 4 |  * |   9 |   2* |   6 | 9·20 + 6·18 = 288


Trace thêm Cách 5 với cùng chuỗi để thấy các trạng thái:

In [19]:
def trace_states(s):
    e0, e1, e2 = 1, 0, 0
    print(f"bắt đầu: e0={e0}, e1={e1}, e2={e2}")
    for c in s:
        if c == '*':
            e0, e1, e2 = (9 * e0 + 9 * e1 + 6 * e2) % MOD, e0, e0
        else:
            e0, e1, e2 = ((c > '0') * e0 + e1 + (c <= '6') * e2) % MOD, (c == '1') * e0, (c == '2') * e0
        print(f"đọc '{c}': e0={e0:>4} (xong), e1={e1:>4} (chờ 1), e2={e2:>4} (chờ 2)")
    return e0

assert trace_states("1*2*") == 288

bắt đầu: e0=1, e1=0, e2=0
đọc '1': e0=   1 (xong), e1=   1 (chờ 1), e2=   0 (chờ 2)
đọc '*': e0=  18 (xong), e1=   1 (chờ 1), e2=   1 (chờ 2)
đọc '2': e0=  20 (xong), e1=   0 (chờ 1), e2=  18 (chờ 2)
đọc '*': e0= 288 (xong), e1=  20 (chờ 1), e2=  20 (chờ 2)


Trace một chuỗi có số 0 để thấy `one('0') = 0` hoạt động: `s = "*0*"`.

In [20]:
r = trace_dp("*0*")
print("brute:", num_decodings_brute("*0*"))
assert r == num_decodings_brute("*0*")

 i | kt | one |  cặp | two | dp[i]
 0 |  - |   - |    - |   - | 1
 1 |  * |   9 |    - |   - | 9
 2 |  0 |   0 |   *0 |   2 | 0·9 + 2·1 = 2
 3 |  * |   9 |   0* |   0 | 9·2 + 0·9 = 18
brute: 18


`"*0"` chỉ có `10` và `20` (2 cách); tiếp theo `*` phải đứng một mình (vì `0*` không ghép được): 2·9 = 18.

## 9. Code nộp LeetCode

Lời giải chính: **DP hai biến** (Cách 4) với hai hàm đếm rõ ràng.

In [21]:
class Solution:
    def numDecodings(self, s: str) -> int:
        MOD = 10**9 + 7

        def one(c: str) -> int:
            # số cách ký tự c đứng một mình thành chữ cái
            if c == '*':
                return 9
            return 0 if c == '0' else 1

        def two(a: str, b: str) -> int:
            # số cách cặp (a, b) thành một số 10..26
            if a == '*':
                if b == '*':
                    return 15            # 11..19, 21..26
                return 2 if b <= '6' else 1
            if a == '1':
                return 9 if b == '*' else 1
            if a == '2':
                if b == '*':
                    return 6             # 21..26
                return 1 if b <= '6' else 0
            return 0

        prev2, prev1 = 1, one(s[0])      # dp[0], dp[1]
        for i in range(1, len(s)):
            cur = (one(s[i]) * prev1 + two(s[i - 1], s[i]) * prev2) % MOD
            prev2, prev1 = prev1, cur
        return prev1

sol = Solution()
assert sol.numDecodings("*") == 9
assert sol.numDecodings("1*") == 18
assert sol.numDecodings("2*") == 15
print("Solution OK")

Solution OK


## 10. Vì sao thuật toán đúng

**Mệnh đề.** `dp[i]` bằng số cặp (cách thay `*` trong `s[0..i-1]`, cách tách `s[0..i-1]` thành các nhóm hợp lệ).

**Chứng minh quy nạp theo `i`.**

- `i = 0`: chuỗi rỗng có đúng một cách (không thay, không tách) → `dp[0] = 1`.
- `i = 1`: một ký tự phải đứng một mình; số giá trị hợp lệ là `one(s[0])`.
- `i >= 2`: chia tập các cặp thành hai nhóm **rời nhau** theo nhóm cuối cùng:
  - **Nhóm cuối dài 1** (chỉ ký tự `s[i-1]`): chọn giá trị cho `s[i-1]` sao cho là chữ số 1..9 (`one(s[i-1])` cách), **độc lập** với phần đầu (`dp[i-1]` cách theo giả thiết quy nạp). Quy tắc nhân: `one · dp[i-1]`.
  - **Nhóm cuối dài 2** (`s[i-2] s[i-1]`): chọn giá trị cho cả hai ký tự sao cho tạo số 10..26 (`two(...)` cách), độc lập với phần đầu `s[0..i-3]` (`dp[i-2]` cách).
  - Hai nhóm rời nhau vì độ dài nhóm cuối khác nhau; hợp lại đủ mọi khả năng. Quy tắc cộng cho công thức.

Lưu ý tính độc lập: một ký tự `*` thuộc **đúng một** nhóm, nên giá trị của nó chỉ được chọn đúng một lần — không đếm trùng.

Modulo không làm sai vì phép cộng và nhân tương thích với `% MOD` (mục 3.1).

## 11. Phân tích Time/Space

| Cách | Time | Space |
|---|---|---|
| 1. Brute force | `O(9^m · n)` | `O(n)` |
| 2. Memo | `O(n)` | `O(n)` (+ stack đệ quy) |
| 3. DP mảng | `O(n)` | `O(n)` |
| 4. Hai biến | `O(n)` | `O(1)` |
| 5. Trạng thái | `O(n)` | `O(1)` |

Mỗi bước chỉ gọi `one`, `two` (vài phép so sánh) → hằng số nhỏ. `n = 10^5` chạy dưới 0,1 giây.

In [22]:
import random
random.seed(3)
big = ''.join(random.choice("0123456789**") for _ in range(10**5))
for name, f in [("dp", num_decodings_dp), ("rolling", num_decodings_rolling), ("states", num_decodings_states), ("Solution", Solution().numDecodings)]:
    t = time.perf_counter()
    r = f(big)
    print(f"{name:>8}: {r}  {time.perf_counter() - t:.3f}s")
big2 = "*" * 10**5
assert num_decodings_rolling(big2) == num_decodings_states(big2) == Solution().numDecodings(big2)
print("chuỗi 10^5 dấu *:", Solution().numDecodings(big2))

      dp: 0  0.020s
 rolling: 0  0.014s
  states: 0  0.010s
Solution: 0  0.014s
chuỗi 10^5 dấu *: 81890275


## 12. Bộ test

Gồm: ví dụ đề, edge case (`"0"`, `"00"`, `"10"`, `"*0"`, `"30"`, `"**"`, chuỗi dài) và đối chiếu ngẫu nhiên có seed giữa mọi cách với brute force.

In [23]:
def all_methods(s):
    out = {
        "memo": num_decodings_memo(s),
        "dp": num_decodings_dp(s),
        "rolling": num_decodings_rolling(s),
        "states": num_decodings_states(s),
        "SolutionMemo": SolutionMemo().numDecodings(s),
        "SolutionDP": SolutionDP().numDecodings(s),
        "SolutionRolling": SolutionRolling().numDecodings(s),
        "SolutionStates": SolutionStates().numDecodings(s),
        "Solution": Solution().numDecodings(s),
    }
    if s.count('*') <= 5:
        out["brute"] = num_decodings_brute(s)
    return out

cases = {
    "*": 9, "1*": 18, "2*": 15,
    "0": 0, "00": 0, "10": 1, "*0": 2, "30": 0, "0*": 0,
    "**": 96, "3*": 9, "*1": 11, "*7": 10, "12": 2, "226": 3, "1": 1,
}
for s, want in cases.items():
    for name, got in all_methods(s).items():
        assert got == want, (s, name, got, want)
print("Tất cả", len(cases), "test cố định OK")

Tất cả 16 test cố định OK


In [24]:
random.seed(2024)
for trial in range(600):
    n = random.randint(1, 8)
    s = ''.join(random.choice("0123456789****") for _ in range(n))
    if s.count('*') > 4:
        continue
    res = all_methods(s)
    want = res["brute"]
    for name, got in res.items():
        assert got == want, (s, name, got, want)
print("Test ngẫu nhiên: mọi cách khớp brute force")

Test ngẫu nhiên: mọi cách khớp brute force


## 13. Lỗi người mới hay gặp

### Lỗi 1: Cho `*` bao gồm cả `0`

**Code sai:** `one('*') = 10`, `two('1','*') = 10`, `two('*','*') = 17` (tính cả 10 và 20).

**Vì sao sai:** đề nói rõ `*` là `1`..`9`.

In [25]:
def two_wrong(a, b):
    if a == '*' and b == '*':
        return 17                      # SAI: tính cả 10, 20
    if a == '1' and b == '*':
        return 10                      # SAI: tính cả 10
    return two(a, b)

def solve_with(two_fn, s):
    prev2, prev1 = 1, one(s[0])
    for i in range(1, len(s)):
        prev2, prev1 = prev1, (one(s[i]) * prev1 + two_fn(s[i - 1], s[i]) * prev2) % MOD
    return prev1

print("sai :", solve_with(two_wrong, "1*"), solve_with(two_wrong, "**"))
print("đúng:", solve_with(two, "1*"), solve_with(two, "**"))

sai : 19 98
đúng: 18 96


**Code đúng:** `two('1','*') = 9`, `two('*','*') = 15` (như hàm `two`).

### Lỗi 2: So sánh `b <= '6'` trước khi xử lý `b == '*'`

**Code sai:** `if a == '2': return 1 if b <= '6' else 0` — với `b = '*'`, vì `'*' < '6'` (mã 42 < 54) nên trả 1 thay vì 6.

In [26]:
def two_wrong2(a, b):
    if a == '2':
        return 1 if b <= '6' else 0    # SAI: quên b == '*'
    return two(a, b)

print("sai :", solve_with(two_wrong2, "2*"), "| đúng:", solve_with(two, "2*"))

sai : 10 | đúng: 15


**Code đúng:** luôn kiểm tra `b == '*'` trước rồi mới so sánh chữ số.

### Lỗi 3: Quên modulo trong vòng lặp (chỉ mod ở cuối)

**Code sai:** cộng nhân không mod, `return dp % MOD` ở cuối.

**Vì sao "sai"?** Kết quả vẫn đúng trong Python (số nguyên lớn), nhưng số có hàng chục nghìn chữ số → chậm khủng khiếp; ở C++/Java thì tràn số cho kết quả sai.

In [27]:
def rolling_no_mod(s):
    prev2, prev1 = 1, one(s[0])
    for i in range(1, len(s)):
        prev2, prev1 = prev1, one(s[i]) * prev1 + two(s[i - 1], s[i]) * prev2   # không mod
    return prev1 % MOD

s_test = "*" * 20000
t = time.perf_counter(); r1 = rolling_no_mod(s_test); t1 = time.perf_counter() - t
t = time.perf_counter(); r2 = num_decodings_rolling(s_test); t2 = time.perf_counter() - t
print(f"không mod: {t1:.3f}s | có mod: {t2:.4f}s | cùng kết quả: {r1 == r2}")

không mod: 0.071s | có mod: 0.0033s | cùng kết quả: True


**Code đúng:** `% MOD` ngay trong phép gán mỗi bước.

### Lỗi 4: Lệch chỉ số giữa `dp[i]` và `s[i]`

**Code sai:** `dp[i] = one(s[i]) * dp[i-1] + two(s[i-1], s[i]) * dp[i-2]` với `dp` độ dài `n + 1` → `IndexError` ở `i = n`, hoặc lệch một ký tự.

**Code đúng:** với `dp[i]` ứng với tiền tố độ dài `i`, ký tự cuối là `s[i-1]`, cặp cuối là `s[i-2], s[i-1]`. Tự hỏi "`dp[i]` nghĩa là gì?" trước khi viết chỉ số.

In [28]:
def dp_wrong_index(s):
    n = len(s)
    dp = [0] * (n + 1)
    dp[0], dp[1] = 1, one(s[0])
    for i in range(2, n + 1):
        try:
            dp[i] = (one(s[i]) * dp[i - 1] + two(s[i - 1], s[i]) * dp[i - 2]) % MOD   # SAI chỉ số
        except IndexError as e:
            return f"IndexError: {e}"
    return dp[n]

print("sai :", dp_wrong_index("1*"), "| đúng:", num_decodings_dp("1*"))

sai : IndexError: string index out of range | đúng: 18


## 14. Mẹo & nhận dạng pattern

- **"Đếm số cách tách chuỗi thành các mảnh hợp lệ, mảnh dài tối đa 2"** → DP `dp[i] = Σ (số cách mảnh cuối) · dp[i - độ dài]`.
- Có **ký tự đại diện** → thay "hợp lệ / không hợp lệ" bằng "**số cách** hợp lệ". Quy tắc cộng–nhân.
- Bảng trường hợp phức tạp → **viết hàm brute nhỏ để kiểm tra bảng** (như `two_brute`).
- Thấy "modulo `10^9+7`" → mod sau **mỗi** phép cộng/nhân.
- `dp[i]` chỉ phụ thuộc vài ô trước → nén còn vài biến.

## 15. Biến thể

1. **Decode Ways I (LC 91)** — không có `*`: `one`, `two` chỉ trả 0/1.
2. **`*` đại diện cho `0`..`9`** — sửa bảng: `one('*') = 9` (0 không đứng một mình), `two('1','*') = 10`, `two('2','*') = 7`, `two('*','*') = 17`, `two('*', b) = 2` nếu `b <= '6'`.
3. **Mã hóa tới 3 chữ số** (ví dụ 1..255) — thêm hạng `three(a, b, c) · dp[i-3]`.
4. **Liệt kê** mọi cách giải mã (thay vì đếm) — backtracking, số lượng có thể rất lớn.
5. **Đếm số cách sinh chuỗi có độ dài n từ bảng chữ cái** — tương tự DP theo vị trí.

In [29]:
def num_decodings_star_with_zero(s):
    """Biến thể 2: * là 0..9."""
    def one2(c):
        return 9 if c == '*' else (0 if c == '0' else 1)
    def two2(a, b):
        A = "0123456789" if a == '*' else a
        B = "0123456789" if b == '*' else b
        return sum(1 for x in A for y in B if x != '0' and 10 <= int(x + y) <= 26)
    prev2, prev1 = 1, one2(s[0])
    for i in range(1, len(s)):
        prev2, prev1 = prev1, (one2(s[i]) * prev1 + two2(s[i - 1], s[i]) * prev2) % MOD
    return prev1

def brute_star_with_zero(s):
    stars = [i for i, c in enumerate(s) if c == '*']
    total = 0
    for choice in product("0123456789", repeat=len(stars)):
        t = list(s)
        for p, d in zip(stars, choice):
            t[p] = d
        total += decode_ways_I(''.join(t))
    return total % MOD

for s in ["1*", "**", "*0*", "2*1", "*"]:
    assert num_decodings_star_with_zero(s) == brute_star_with_zero(s)
print("Biến thể 2:", {s: num_decodings_star_with_zero(s) for s in ["1*", "2*", "**"]})

Biến thể 2: {'1*': 19, '2*': 16, '**': 98}


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | thay mọi `*`, chạy LC 91 | `O(9^m · n)` | `O(n)` | trọng tài kiểm thử |
| 2. Memo | `f(i)` hậu tố, đệ quy | `O(n)` | `O(n)` | khi nghĩ tự nhiên theo đệ quy; cẩn thận độ sâu |
| 3. DP mảng | `dp[i]` tiền tố | `O(n)` | `O(n)` | dễ trace, dễ debug |
| 4. Hai biến | nén DP | `O(n)` | `O(1)` | bản nộp chuẩn |
| 5. Trạng thái | `e0/e1/e2` chờ ghép | `O(n)` | `O(1)` | gọn, tổng quát cho máy trạng thái |

## 17. Bài tập

**Bài 1 (đoán output).** Không chạy code: `numDecodings("*1")`, `numDecodings("3*")`, `numDecodings("**0")`?

**Bài 2 (tính tay).** Lập bảng `dp` như mục 8 cho `s = "*2*1"`.

**Bài 3 (điền TODO).** Hoàn thiện `two_my(a, b)` (không nhìn bảng) và tự kiểm tra bằng `two_brute`.

**Bài 4 (tự viết lại).** Viết lại Cách 5 (máy trạng thái) `my_states(s)` mà không nhìn code.

**Bài 5 (mini challenge).** Viết `count_three(s)`: đếm số cách giải mã khi bảng mã là `1..255` (mỗi nhóm 1–3 chữ số, không có 0 đứng đầu), `s` chỉ có chữ số (không `*`), mod `10^9+7`.

In [30]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def two_my(a, b):
    # TODO: xử lý a == '*', a == '1', a == '2'; nhớ kiểm tra b == '*' trước
    pass

# Bài 4 — TODO
def my_states(s):
    # TODO: e0, e1, e2 = 1, 0, 0; cập nhật theo c == '*' hay chữ số
    pass

# Bài 5 — TODO
def count_three(s):
    # TODO: dp[i] = dp[i-1]*(1 chữ số hợp lệ) + dp[i-2]*(2 chữ số) + dp[i-3]*(3 chữ số <= 255)
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: dùng công thức `dp[2] = one(b)·dp[1] + two(a,b)·dp[0]`.
- Mức 2: `"*1"`: `9·1 + two('*','1')·1 = 9 + 2`; `"3*"`: `9·1 + 0`.
- Mức 3: `"**0"`: `dp[2] = 96`; `dp[3] = 0·96 + two('*','0')·9 = 2·9 = 18`.

**Bài 2.**
- Mức 1: `dp[0] = 1, dp[1] = 9`.
- Mức 2: `dp[2] = 1·9 + two('*','2')·1 = 11`; `dp[3] = 9·11 + two('2','*')·9`.
- Mức 3: `dp[3] = 99 + 54 = 153`; `dp[4] = 1·153 + two('*','1')·11 = 153 + 22 = 175`.

**Bài 3.**
- Mức 1: `a == '*'`: ba trường hợp của `b`.
- Mức 2: `a == '1'`: 9 hoặc 1; `a == '2'`: 6, 1 hoặc 0.
- Mức 3: còn lại trả 0.

**Bài 4.**
- Mức 1: ba biến `e0, e1, e2`.
- Mức 2: `*`: `9e0 + 9e1 + 6e2`, `e0`, `e0`.
- Mức 3: chữ số: `(c>'0')e0 + e1 + (c<='6')e2`, `(c=='1')e0`, `(c=='2')e0`.

**Bài 5.**
- Mức 1: `dp[0] = 1`.
- Mức 2: nhóm 2 chữ số hợp lệ nếu chữ số đầu khác 0; nhóm 3 chữ số hợp lệ nếu chữ số đầu khác 0 và `<= 255`.
- Mức 3: kiểm chứng bằng đệ quy liệt kê trên chuỗi ngắn.

## 19. Đáp án tham khảo

In [31]:
# Bài 1
assert Solution().numDecodings("*1") == 11
assert Solution().numDecodings("3*") == 9
assert Solution().numDecodings("**0") == 18

# Bài 2
trace_dp("*2*1")
assert Solution().numDecodings("*2*1") == 175 == num_decodings_brute("*2*1")

# Bài 3
def two_my_ans(a, b):
    if a == '*':
        if b == '*':
            return 15
        return 2 if b <= '6' else 1
    if a == '1':
        return 9 if b == '*' else 1
    if a == '2':
        if b == '*':
            return 6
        return 1 if b <= '6' else 0
    return 0

assert all(two_my_ans(a, b) == two_brute(a, b) for a in "0123456789*" for b in "0123456789*")
print("Bài 1–3 OK")

 i | kt | one |  cặp | two | dp[i]
 0 |  - |   - |    - |   - | 1
 1 |  * |   9 |    - |   - | 9
 2 |  2 |   1 |   *2 |   2 | 1·9 + 2·1 = 11
 3 |  * |   9 |   2* |   6 | 9·11 + 6·9 = 153
 4 |  1 |   1 |   *1 |   2 | 1·153 + 2·11 = 175
Bài 1–3 OK


In [32]:
# Bài 4
def my_states_ans(s):
    e0, e1, e2 = 1, 0, 0
    for c in s:
        if c == '*':
            e0, e1, e2 = (9 * e0 + 9 * e1 + 6 * e2) % MOD, e0, e0
        else:
            e0, e1, e2 = ((c > '0') * e0 + e1 + (c <= '6') * e2) % MOD, (c == '1') * e0, (c == '2') * e0
    return e0

# Bài 5
def count_three_ans(s):
    n = len(s)
    dp = [0] * (n + 1)
    dp[0] = 1
    for i in range(1, n + 1):
        if s[i - 1] != '0':
            dp[i] += dp[i - 1]
        if i >= 2 and s[i - 2] != '0':
            dp[i] += dp[i - 2]
        if i >= 3 and s[i - 3] != '0' and int(s[i - 3:i]) <= 255:
            dp[i] += dp[i - 3]
        dp[i] %= MOD
    return dp[n]

def count_three_brute(s):
    if not s:
        return 1
    total = 0
    for L in (1, 2, 3):
        if L <= len(s) and s[0] != '0' and int(s[:L]) <= 255:
            total += count_three_brute(s[L:])
    return total

random.seed(9)
for _ in range(300):
    s = ''.join(random.choice("0123456789**") for _ in range(random.randint(1, 7)))
    if s.count('*') <= 4:
        assert my_states_ans(s) == num_decodings_brute(s)
    d = ''.join(random.choice("0123456789") for _ in range(random.randint(1, 10)))
    assert count_three_ans(d) == count_three_brute(d) % MOD
print("Bài 4–5 OK (đối chiếu brute force)")

Bài 4–5 OK (đối chiếu brute force)


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template DP đếm cách tách chuỗi (mảnh 1–2 ký tự):**

```python
prev2, prev1 = 1, one(s[0])          # dp[0], dp[1]
for i in range(1, len(s)):
    cur = (one(s[i]) * prev1 + two(s[i-1], s[i]) * prev2) % MOD
    prev2, prev1 = prev1, cur
return prev1
```

**Bảng cần thuộc:** `one: '*'→9, '0'→0, khác→1`; `two: '**'→15, '*d'→2 (d≤6) / 1, '1*'→9, '2*'→6, '1d'→1, '2d'→1 (d≤6) / 0, còn lại 0`.

**Checklist:**

- [ ] `*` là `1..9`, **không** có `0`.
- [ ] Kiểm tra `b == '*'` **trước** khi so sánh `b <= '6'`.
- [ ] `dp[0] = 1` (chuỗi rỗng), `dp[1] = one(s[0])`.
- [ ] `% MOD` ở mỗi bước.
- [ ] Chỉ số: `dp[i]` ↔ ký tự cuối `s[i-1]`.
- [ ] Thử `"0"`, `"*0"`, `"30"`, `"**"`, chuỗi `10^5` ký tự.